## SAMoSA Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [ ]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import sys, re, json
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle

# Dynamically find repo root (works from any notebook location)
ROOT = Path.cwd()
while not (ROOT / "UTILS").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from UTILS.helpers import (
    convert_unit, _canon,
    load_contracts, to_continuous_stream, run_qa_checks, check_sample_integrity,
)

C       = load_contracts(ROOT)
SCHEMA  = C["SCHEMA"]
RAW2ID  = C["RAW2ID"]
ID2NAME = C["ID2NAME"]
UNKNOWN_ID = C["UNKNOWN_ID"]
TARGET_HZ  = C["TARGET_HZ"]
CLEANED    = C["CLEANED"]
MERGED     = C["MERGED"]

RAW = ROOT / "data" / "raw_data" / "SAMoSA" / "TrainingDataset"

print("Paths & contracts ready.")
print("Schema keys:", list(SCHEMA.keys()))
print("RAW dir :", RAW)
print("CLEANED :", CLEANED)
print("MERGED  :", MERGED)

act_map = json.loads((ROOT / "Unification" / "schemas" / "activity_mapping.json").read_text())
print(f"Activity mapping loaded. Number of activities: {len(act_map['label_set'])}")

### Step 1. Ingest, preporccess and map the data 

In [ ]:
# ====================================
# STEP 1 — Ingest & preprocess SAMoSA
# ====================================
def load_samosa_raw(
    raw_dir: Path,
    *,
    sampling_rate_hz: float = 50.0,
    axis_remap: Optional[Dict[str, str]] = None,   # optional FLU correction
) -> pd.DataFrame:
    """
    Load all SAMoSA TrainingDataset .pkl files and return an intermediate frame:

        subject_id (str)   e.g. "S07"
        session_id (str)   per-(context,trial) session
        timestamp_s (float; synthetic, 1/50 steps from 0)
        acc_x/y/z (float32, m/s^2, gravity-included)
        gyro_x/y/z (float32, rad/s)
        activity_label_raw (str; verbatim from filename)
        context_raw (str; verbatim from filename)

    We ignore audio entirely and drop orientation channels.
    """
    raw_dir = Path(raw_dir)
    assert raw_dir.exists(), f"SAMoSA RAW path not found: {raw_dir}"

    files = _collect_samosa_files(raw_dir)
    if not files:
        print("[SAMoSA] No .pkl files found.")
        return pd.DataFrame()

    dataset_name = "samosa"
    all_rows: list[pd.DataFrame] = []

    # Optional axis-remap maps canonical names → canonical names
    acc_names  = ["acc_x", "acc_y", "acc_z"]
    gyro_names = ["gyro_x", "gyro_y", "gyro_z"]
    acc_map  = {k: v for k, v in (axis_remap or {}).items() if k.startswith("acc_")}
    gyro_map = {k: v for k, v in (axis_remap or {}).items() if k.startswith("gyro_")}

    for fp in tqdm(files, desc="[SAMoSA] files"):
        stem = fp.stem  # e.g. "49---Kitchen---Chopping---1"
        try:
            pid_str, context, activity, trial_str = stem.split("---")
        except ValueError:
            print(f"[WARN] Unexpected filename format: {fp.name} — skipping.")
            continue

        # Subject ID
        try:
            pid_int = int(pid_str)
            subject_id = f"S{pid_int:02d}"
        except Exception:
            subject_id = f"S{abs(hash(pid_str)) % 100000:05d}"

        # Trial/session identifier
        try:
            trial_id = int(trial_str)
        except Exception:
            trial_id = 1

        context_raw   = context.strip()
        activity_raw  = activity.strip()
        context_norm  = _canon(context_raw)
        activity_norm = _canon(activity_raw)

        session_id = f"{context_norm}_{activity_norm}_t{trial_id:02d}"

        # Load pickle and extract IMU
        try:
            with open(fp, "rb") as f:
                obj = pickle.load(f)
        except Exception as e:
            print(f"[WARN] Failed to read {fp.name}: {e} — skipping.")
            continue

        if "IMU" not in obj:
            print(f"[WARN] {fp.name}: no 'IMU' key — skipping.")
            continue

        imu = np.asarray(obj["IMU"], dtype=np.float64)
        if imu.ndim != 2 or imu.shape[1] < 6:
            print(f"[WARN] {fp.name}: unexpected IMU shape {imu.shape} — skipping.")
            continue

        # Columns: [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, orient(3)]
        acc  = imu[:, 0:3]
        gyro = imu[:, 3:6]

        # Axis remap hook (to enforce FLU once known)
        if acc_map:
            acc = _apply_axis_map(acc, acc_names, acc_map)
        if gyro_map:
            gyro = _apply_axis_map(gyro, gyro_names, gyro_map)

        # Synthetic timestamps at 50 Hz (per-session)
        N = imu.shape[0]
        timestamps_s = np.arange(N, dtype=np.float64) / float(sampling_rate_hz)

        df = pd.DataFrame({
            "subject_id":        subject_id,
            "session_id":        session_id,
            "timestamp_s":       timestamps_s,

            "acc_x": acc[:, 0],
            "acc_y": acc[:, 1],
            "acc_z": acc[:, 2],
            "gyro_x": gyro[:, 0],
            "gyro_y": gyro[:, 1],
            "gyro_z": gyro[:, 2],

            "activity_label_raw": activity_raw,
            "context_raw":        context_raw,
        })

        df["subject_id"] = df["subject_id"].astype("string")
        df["session_id"] = df["session_id"].astype("string")
        all_rows.append(df)

    if not all_rows:
        return pd.DataFrame()

    raw = pd.concat(all_rows, ignore_index=True)

    print("\n=== RAW SUMMARY (SAMoSA) ===")
    print(f"Rows: {raw.shape[0]:,}")
    print(f"Estimated Hz: ~{_estimate_hz_from_index(len(raw)):0.2f}")
    print("Top native labels (verbatim):")
    print(raw["activity_label_raw"].value_counts(dropna=False).head(20))
    print("\nContexts:")
    print(raw["context_raw"].value_counts(dropna=False))

    return raw


raw_samosa = load_samosa_raw(RAW, sampling_rate_hz=50.0, axis_remap=None)
raw_samosa.head(3)

### Step 2. Map the data and audit the mapping

In [ ]:
# =========================================================
# STEP 2 — Audit raw_label → native_id → global_id mapping
# =========================================================

if raw_samosa.empty:
    raise SystemExit("No SAMoSA rows after loading. Check RAW path/layout.")

# Normalize labels to mapping keys
raw_counts = (
    raw_samosa["activity_label_raw"]
        .astype(str).map(_canon)
        .value_counts()
        .rename_axis("raw_label")
        .reset_index(name="count")
)

# Native ID map for this dataset (keep UNKNOWN_ID reserved for 'unknown')
labels_sorted = sorted([rl for rl in raw_counts["raw_label"].unique()
                        if rl != _canon("unknown")])

NATIVE_LBL2ID = {_canon("unknown"): UNKNOWN_ID}
NATIVE_LBL2ID.update({lbl: i for i, lbl in enumerate(labels_sorted, start=0)})

raw_counts["native_id"]  = raw_counts["raw_label"].map(NATIVE_LBL2ID).astype(int)
raw_counts["mapped_gid"] = raw_counts["raw_label"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"]  = raw_counts["mapped_gid"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_gid"] == UNKNOWN_ID]

print(f"SAMoSA raw label unique: {len(raw_counts)} | Unmapped→global: {len(unmapped)}")
print("Unmapped (top-15):")
print(unmapped.nlargest(15, "count")[["raw_label","count"]].to_string(index=False))

raw_counts.head(27)

### Step 3. Build and clean dataset in stream json fromat

In [ ]:
# =========================================================
# STEP 3 — Build schema-ordered continuous_stream SAMoSA df
# =========================================================
# Adapt Samosa's native columns to standard format
adapted = raw_samosa.copy()
adapted["dataset"] = "samosa"
adapted["timestamp_ns"] = (adapted["timestamp_s"].astype(np.float64) * 1e9).round().astype("int64")
adapted = adapted.rename(columns={"activity_label_raw": "dataset_activity_label"})

# Assign native IDs from NATIVE_LBL2ID built in Step 2
adapted["dataset_activity_id"] = (
    adapted["dataset_activity_label"].astype(str).map(_canon)
    .map(NATIVE_LBL2ID).fillna(UNKNOWN_ID).astype("int16")
)

samosa_df = to_continuous_stream(adapted, SCHEMA, RAW2ID, ID2NAME, UNKNOWN_ID)
print("UNIFIED rows:", len(samosa_df))
samosa_df.head(3)

### Step 4. Audit check the unified frame

In [ ]:
# ==========================================
# STEP 4 — Contract checks & quick QA
# ==========================================
run_qa_checks(samosa_df, SCHEMA, UNKNOWN_ID)
check_sample_integrity(samosa_df, SCHEMA)

### Step 5. Save outputs

In [ ]:
# ===============================
# STEP 5 — Save cleaned SAMoSA
# ===============================
CLEANED.mkdir(parents=True, exist_ok=True)
out_path = CLEANED / "samosa_clean_data.parquet"
samosa_df.to_parquet(out_path, index=False)
print("Saved:", out_path)
